In [24]:
import config
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_cerebras import ChatCerebras
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import ConversationalRetrievalChain
# Updated imports for modern LangChain
# Updated imports for LangChain 2026
from langchain_classic.chains import (
    create_history_aware_retriever, 
    create_retrieval_chain
)
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [33]:
url = "https://365datascience.com/courses"

In [34]:
loader = WebBaseLoader(url)

In [35]:
raw_documents = loader.load()

In [36]:
text_splitter = RecursiveCharacterTextSplitter()
documents = text_splitter.split_documents(raw_documents)

In [37]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [38]:
ids = [f"chunk_{i}" for i in range(len(documents))]

# Pass the ids list directly to FAISS
vectorstore = FAISS.from_documents(
    documents=documents,
    embedding=embeddings,
    ids=ids
)

In [39]:
from langchain.memory import ConversationBufferMemory

In [40]:
memory = ConversationBufferMemory(
    memory_key = 'chat_history',
    return_messages = True
)

In [41]:
llm = ChatCerebras(
    model='llama3.1-8b',
    api_key = config.apikey,
    temperature = 0
)

In [42]:
# 1. Define the system prompt for the answer chain
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, try to find it. If not found say you don't know and could not find anything "
    "Use three sentences maximum and keep the answer concise.\n\n"
    "{context}"
)

# 2. Create the prompt templates
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

# 3. Create the document processing chain
# Note: This replaces the 'stuff' chain type from your original code
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# 4. Create the final RAG chain
rag_chain = create_retrieval_chain(vectorstore.as_retriever(), question_answer_chain)

# 5. Execute a query
# Note: History-aware chains use 'input' instead of 'question'
response = rag_chain.invoke({
    "input": "What are the upcoming courses?", 
    "chat_history": memory.load_memory_variables({})["chat_history"]
})

print(response["answer"])

I don't know and could not find anything about upcoming courses in the provided context.


In [43]:
response = rag_chain.invoke({
    "input": "what are the courses on the site and what are the latest ones?",
    "chat_history": memory.load_memory_variables({})["chat_history"]
})
print(response["answer"])

The courses on the site include topics such as Data Science, Machine Learning, AI, Data Engineering, Business Skills, Finance Skills, and more. Some of the specific courses listed include:

* Data Science courses: Introduction to Data and Data Science, Intro to AI, AI Strategy, Data Strategy, Introduction to Data Architecture, Data Analysis with Excel Pivot Tables, and Advanced Microsoft Excel.
* Machine Learning courses: Deep Learning with Pytorch, Deep Learning with TensorFlow 2, Convolutional Neural Networks with TensorFlow in Python, and Speech Recognition with Python.
* AI courses: Intro to AI, AI Strategy, Intro to AI Agents and Agentic AI, and AI Ethics.
* Business Skills courses: Communication and Presentation Skills for Analysts and Managers, Marketing Strategy, Negotiation, and Management.
* Finance Skills courses: Python for Finance, Introduction to Business Analytics, and Quant Finance Analyst.
* Data Engineering courses: Building Data Pipelines with Apache Airflow, Data In